## Setup and imports

In [ ]:
EXP_NAME = "10MAY_simple_synth_biased_2"
dataset = "synth/9MAY/simple_biased_train"
feature_map = "sps/features_synth_simple_biased"
candidates = ['biomarker_1', 'biomarker_2_obs']

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'

In [ ]:
print(PROJECT_ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.feature_selection import mutual_info_regression
from sklearn.utils import resample

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.2)

sps_audit = pd.read_csv(f'{PROJECT_ROOT}/results/{EXP_NAME}/sps_audit.csv')
sbs_audit_baseline = pd.read_csv(f'{PROJECT_ROOT}/results/{EXP_NAME}/sps_audit_baseline.csv')
dataset = pd.read_csv(f'{PROJECT_ROOT}/data/{dataset}.csv')

with open(f"{PROJECT_ROOT}/configs/{feature_map}.json", 'r') as f:
  features = json.load(f)

In [ ]:
group_mace_cols = sps_audit.columns.str.startswith('mace_')
group_mace_baseline_cols = sbs_audit_baseline.columns.str.startswith('mace_')
sps_audit['max_mace'] = sps_audit.loc[:, group_mace_cols].max(axis=1)
sbs_audit_baseline['max_mace'] = sbs_audit_baseline.loc[:, group_mace_baseline_cols].max(axis=1)

In [ ]:
full_audit = pd.concat([sps_audit, sbs_audit_baseline], ignore_index=True, axis=0)
tradeoff_audit = full_audit.groupby('iteration')[['te_error', 'max_mace', 'auprc']].median().round(3)
x_desc_configs = full_audit[full_audit['bucket'] == 'x_desc'].groupby(['iteration'])['feature'].apply(set).to_dict()


In [ ]:
positives = dataset[dataset[features["target"]["name"]] == 1]
auprc_baseline = len(positives) / len(dataset)
print(f'Chance-level AUPRC: {auprc_baseline:.3f}')

In [ ]:
# Correlation between candidates

corr_matrix = dataset[candidates].corr(method="spearman")

print(corr_matrix)


In [ ]:
print(tradeoff_audit)

In [ ]:
configs_dict = {frozenset(v): k for k, v in x_desc_configs.items()}

all_candidates_config = configs_dict.get(frozenset(candidates))


feature_analysis = []
for feature in candidates:
  other_candidates = candidates.copy()
  other_candidates.remove(feature)
  ablation_config = configs_dict.get(frozenset(other_candidates))
  delta_auprc = (tradeoff_audit.loc[all_candidates_config, 'auprc'] - tradeoff_audit.loc[ablation_config, 'auprc']) / (tradeoff_audit.loc[ablation_config, 'auprc'] - auprc_baseline)
  delta_mace = (tradeoff_audit.loc[all_candidates_config, 'max_mace'] - tradeoff_audit.loc[ablation_config, 'max_mace']) / tradeoff_audit.loc[ablation_config, 'max_mace']
  feu = delta_mace / delta_auprc
  feature_analysis.append({
    'feature': feature,
    'delta_auprc': delta_auprc,
    'delta_mace': delta_mace,
    'feu': feu
  })

feature_analysis_df = pd.DataFrame(feature_analysis)
print(feature_analysis_df.to_markdown())
